# 🌟 COSMIC v0.0.1 - Compatibilidad Automática

**Este notebook ha sido actualizado para COSMIC v0.0.1**

## ✨ Mejoras Disponibles
- **Estructura modular**: `cosmic.core`, `cosmic.io`, `cosmic.preprocess`, `cosmic.analysis`
- **Imports modernos**: Específicos y organizados
- **API mejorada**: Mejor manejo de errores y configuración
- **Compatibilidad total**: El código existente sigue funcionando

## 🔄 Migración Opcional
Para usar las nuevas funcionalidades:
```python
# Nuevo (recomendado)
from cosmic.analysis.analyzer import ClusterAnalyzer
from cosmic.core.clustering import Clustering
from cosmic.io.loader import DataLoader

# Legacy (sigue funcionando)
from COSMIC import ClusterAnalyzer
from clustering import HDBSCANClustering
from data_loader import DataLoader
```

📚 **Documentación**: `/docs/reference/NOTEBOOK_MIGRATION_GUIDE.md`

In [ ]:
from astropy.io import ascii
from astropy.table import QTable,join
from astropy.coordinates import SkyCoord, Galactocentric, Angle
from astropy.visualization import quantity_support
import hdbscan
import seaborn as sns
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from zero_point import zpt
import matplotlib.ticker as ticker
from astropy.coordinates import SkyCoord
zpt.load_tables()

quantity_support()
%matplotlib inline
%config InlineBackend.figure_format ='retina'

In [ ]:
# Import data
data_gaia = QTable.read('25_arcmin_GAIA_DR3.ecsv',guess=False,format='ascii.ecsv')
print(f'The data constains {len(data_gaia)} sources')
fidelity_condition = (data_gaia['fidelity_v2'] >= 0.5)
data_gaia = data_gaia[fidelity_condition]
print(f'Only {len(data_gaia)} sources with fidelity above 0.5')
data_tmass =  QTable.read('NGC6383_40arcmin_2MASS.ecsv',guess=False,format='ascii.ecsv')
print(f"There's {len(data_tmass)} stars with 2MASS data")
data_tmass.rename_columns(['ra', 'dec','designation'],['ra_tmass','dec_tmass','designation_tmass'])
data_tmass = data_tmass[data_tmass['angular_distance'] <= 0.3*u.arcsec]
print(f"There's {len(data_tmass)} stars with 2MASS data with a separation below 0.3 arcsec")
data_gaia = join(data_gaia,data_tmass,keys='source_id',join_type='left')

In [ ]:
# # Edit table columns
data_gaia.rename_columns(['phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag', 'bp_rp'],['Gmag', 'G_BPmag', 'G_RPmag', 'BP-RP'])

In [ ]:
columns_to_check = ['ra','dec','pmra','pmdec','parallax','Gmag','G_BPmag','G_RPmag', 'l', 'b']
mask = np.zeros(len(data_gaia), dtype=bool)
for column in columns_to_check:
    mask |= ~np.isfinite(data_gaia[column])

# Apply the mask to the QTable
data_gaia = data_gaia[~mask]
print(f"There's only {len(data_gaia)} sources with valid data")

In [ ]:
# # Zero-Point Parallax
data_gaia.rename_columns(['parallax'],['parallax_observed']);
data_gaia['zpvals'] = zpt.get_zpt(data_gaia['Gmag'], data_gaia['nu_eff_used_in_astrometry'], data_gaia['pseudocolour'], data_gaia['ecl_lat'], data_gaia['astrometric_params_solved'])*u.mas
data_gaia['zpvals'] = np.ma.masked_invalid(data_gaia['zpvals']).filled(0)
data_gaia['parallax'] = data_gaia['parallax_observed'] - data_gaia['zpvals']

In [ ]:
data_gaia['pmra_obs'],data_gaia['pmdec_obs'] = data_gaia['pmra'],data_gaia['pmdec']

In [ ]:
# Correct proper motion to align with ICRF
def edr3ToICRF(pmra ,pmdec ,ra ,dec ,G):
    if G >=13:
        return pmra , pmdec
    def sind (x):
        return np.sin(np. radians (x))
    def cosd (x):
        return np.cos(np. radians (x))
    table1 =""" 0.0 9.0 18.4 33.8 -11.3
                9.0 9.5 14.0 30.7 -19.4
                9.5 10.0 12.8 31.4 -11.8
                10.0 10.5 13.6 35.7 -10.5
                10.5 11.0 16.2 50.0 2.1
                11.0 11.5 19.4 59.9 0.2
                11.5 11.75 21.8 64.2 1.0
                11.75 12.0 17.7 65.6 -1.9
                12.0 12.25 21.3 74.8 2.1
                12.25 12.5 25.7 73.6 1.0
                12.5 12.75 27.3 76.6 0.5
                12.75 13.0 34.9 68.9 -2.9 """
    table1 = np.fromstring(table1,sep=' ').reshape((12,5)).T
    Gmin = table1[0]
    Gmax = table1[1]
    # pick the appropriate omegaXYZ for the source ’s magnitude :
    omegaX = table1[2][(Gmin <=G)&(Gmax>G)][0]
    omegaY = table1[3][(Gmin <=G)&(Gmax>G)][0]
    omegaZ = table1[4][(Gmin <=G)&(Gmax>G)][0]
    pmraCorr = -1*sind(dec)*cosd(ra)*omegaX-sind(dec)*sind(ra)*omegaY + cosd(dec)*omegaZ
    pmdecCorr = sind(ra)*omegaX-cosd(ra)*omegaY
    return pmra - pmraCorr/1000. , pmdec - pmdecCorr/1000.

In [ ]:
for i in data_gaia:
    i['pmra'],i['pmdec'] = edr3ToICRF(i['pmra_obs'].value,i['pmdec_obs'].value,i['ra'].value,i['dec'].value,i['Gmag'].value)*(u.mas/u.yr)

In [ ]:
data_gaia['ra_tmass','dec_tmass','angular_distance'].filled(np.nan);

In [ ]:
def add_photometric_errors(table):
    """
    Adds photometric errors for Gmag, G_BPmag, G_RPmag, and e_bp_rp to the given QTable,
    incorporating astropy units (u.mag) for proper handling of magnitudes, with errors expressed in milli-magnitudes.

    Parameters:
    - table: QTable, must contain columns 'Gmag', 'G_BPmag', and 'G_RPmag' for G-band magnitudes.

    The function modifies the table in place by adding four new columns for the errors:
    - 'e_Gmag', 'e_G_BPmag', 'e_G_RPmag', 'e_BP_RP', all expressed in magnitudes.
    """
    # Define error functions for g_mag, g_bp, g_rp with corrections for units
    def g_mag_error(G):
        G_val = G.to(u.mag).value  # Ensure G is in magnitudes and get its value
        if G_val < 13:
            return 0.3 / 1000 * u.mag  # Convert mmag to mag
        elif G_val < 17:
            return np.interp(G_val, [13, 17], [0.3, 1]) / 1000 * u.mag
        elif G_val <= 20:
            return np.interp(G_val, [17, 20], [1, 6]) / 1000 * u.mag
        else:
            return 6 / 1000 * u.mag
    
    def g_bp_error(G):
        G_val = G.to(u.mag).value
        if G_val < 13:
            return 0.9 / 1000 * u.mag
        elif G_val < 17:
            return np.interp(G_val, [13, 17], [0.9, 12]) / 1000 * u.mag
        elif G_val <= 20:
            return np.interp(G_val, [17, 20], [12, 108]) / 1000 * u.mag
        else:
            return 108 / 1000 * u.mag
    
    def g_rp_error(G):
        G_val = G.to(u.mag).value
        if G_val < 13:
            return 0.6 / 1000 * u.mag
        elif G_val < 17:
            return np.interp(G_val, [13, 17], [0.6, 6]) / 1000 * u.mag
        elif G_val <= 20:
            return np.interp(G_val, [17, 20], [6, 52]) / 1000 * u.mag
        else:
            return 52 / 1000 * u.mag
    
    # Apply the error functions to the respective G column to create new error columns with units
    table['e_Gmag'] = [g_mag_error(G) for G in table['Gmag']]
    table['e_G_BPmag'] = [g_bp_error(G) for G in table['G_BPmag']]
    table['e_G_RPmag'] = [g_rp_error(G) for G in table['G_RPmag']]
    
    # Calculate e_BP_RP as the square root of the sum of squares of e_G_BPmag and e_G_RPmag, correctly applying units
    table['e_BP_RP'] = np.sqrt(table['e_G_BPmag']**2 + table['e_G_RPmag']**2)

In [ ]:
add_photometric_errors(data_gaia)

In [ ]:
n = []
past_len = np.inf
min_cluster_size_samples = np.arange(10,300,1)
effective_mcs = []
lambda_value = []
for i in min_cluster_size_samples:
    clusterer = hdbscan.HDBSCAN(algorithm='best',
                            cluster_selection_method='leaf',
                            allow_single_cluster=True,
                            min_cluster_size=i,
                            core_dist_n_jobs=-1,
                            gen_min_span_tree=True,
                            metric='euclidean',
                           ).fit(data_gaia['pmra','pmdec'].to_pandas())
    tree = clusterer.condensed_tree_.to_pandas()
    # Find the index of the row with the maximum lambda_val
    if (tree["lambda_val"].max() >= 8):
        max_lambda_val_row = tree["lambda_val"].idxmax()
        # Retrieve the parent value for the row with the maximum lambda_val
        desired_parent = tree.at[max_lambda_val_row, "parent"]
        desired_len = len(tree[tree["parent"] == desired_parent])
        if (desired_len < 700) and (len(np.unique(clusterer.labels_)) > 1) and (desired_len >= 200):
            n.append(desired_len)
            effective_mcs.append(i)
            lambda_value.append(tree["lambda_val"].max())

In [ ]:
# Find the index of the maximum value in n for plotting the vertical line
max_n_value = max(n)
max_n_index = n.index(max_n_value)  # Guard against empty list
max_min_cluster_size = effective_mcs[max_n_index]

max_lambda_value = lambda_value[max_n_index]
max_lambda_index = lambda_value.index(max_lambda_value)
max_lambda_min_cluster_size = effective_mcs[max_lambda_index]

# Plotting
fig,ax = plt.subplots(layout='constrained',figsize=(6,5))
ax.plot(effective_mcs, n)
ax.axvline(max_min_cluster_size, color='b', linestyle='--', label=f'Optimal Min Cluster Size: {max_min_cluster_size}')
ax.axhline(y=max_n_value, color='g', linestyle='--', label=f'Max Cluster Size: {max_n_value}')
#ax.axvline(min_min_cluster_size, color='r', linestyle='--', label=f'Optimal Min Cluster Size: {min_min_cluster_size}')
#ax.axhline(y=min_n_value, color='fuchsia', linestyle='--', label=f'Max Cluster Size: {min_n_value}')
ax.set_xlabel('Min Cluster Size')
ax.set_ylabel('Cluster Size')
ax.set_title('Cluster Size as a Function of Min Cluster Size')
ax.legend()

fig.savefig('Tex_File/Figures/min_cluster_size.pdf',bbox_inches='tight')
plt.show()

In [ ]:
clusterer = hdbscan.HDBSCAN(algorithm='best',
                            cluster_selection_method='leaf',
                            allow_single_cluster=True,
                            min_cluster_size=43,#min_cluster_size_samples_used[max_n_index],
                            #min_cluster_size=20,
                            core_dist_n_jobs=-1,
                            gen_min_span_tree=True,
                            metric='euclidean',
                           ).fit(data_gaia['pmra','pmdec'].to_pandas())

In [ ]:
data_gaia['cluster'] = clusterer.labels_
data_gaia['probability'] = clusterer.probabilities_
color_palette = sns.color_palette('bright',len(np.unique(clusterer.labels_)))
data_gaia['cluster_member_colors'] = [color_palette[x] if x >= 0 else (0.5, 0.5, 0.5) for x in clusterer.labels_]
data_gaia['outlier_score'] = clusterer.outlier_scores_

In [ ]:
tree = clusterer.condensed_tree_.to_pandas()
# Find the index of the row with the maximum lambda_val
max_lambda_val_row = tree["lambda_val"].idxmax()
# Retrieve the parent value for the row with the maximum lambda_val
desired_parent = tree.at[max_lambda_val_row, "parent"]
desired_len = len(tree[tree["parent"] == desired_parent])

In [ ]:
desired_len

In [ ]:
### Crea el condensed cluster tree.

fig_ct, ax_ct = plt.subplots(1,1, layout='constrained',figsize=(8,6))
clusterer.condensed_tree_.plot(select_clusters=True,selection_palette=sns.color_palette('bright',len(np.unique(clusterer.labels_))),cmap=sns.color_palette("mako", as_cmap=True),axis=ax_ct)
ax_ct.set(ylabel=r'$\lambda$ value')
ax_ct.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_ct.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_ct.tick_params(axis='both', which='both', direction='in')
fig_ct.savefig('Tex_File/Figures/condensed_cluster_tree_NGC6383.pdf');

In [ ]:
# Calculate the size of each cluster in data_gaia_qtable
# QTable does not have the value_counts method, so we'll use a different approach
clusters, counts = np.unique(data_gaia["cluster"], return_counts=True)

# Find the cluster(s) with the same size as the desired cluster size
matching_clusters = clusters[counts == desired_len]

# For simplicity, assuming there's only one matching cluster and getting its name
desired_cluster_name_qtable = matching_clusters[0] if len(matching_clusters) > 0 else None

desired_cluster_name_qtable, len(matching_clusters)  # Returning the cluster name and the number of matching clusters

In [ ]:
data_gaia['cluster_hdbscan'] = data_gaia['cluster']

In [ ]:
data_gaia['cluster'] = np.where(data_gaia['cluster'] == desired_cluster_name_qtable, 
                                       data_gaia['cluster'], 
                                       -1)

In [ ]:
len(data_gaia[data_gaia['cluster'] == 0])

In [ ]:
ascii.write(data_gaia,'25_arcmin_clustered.ecsv',format='ecsv',overwrite=True) # Create the clustered file

## Selected cluster plots

In [ ]:
data_gaia_selected = data_gaia[raw_cluster]['l','b','pmra','pmdec_corr','parallax_corrected'].to_pandas()

In [ ]:
clusterer_selected = hdbscan.HDBSCAN(algorithm='best', metric='euclidean',allow_single_cluster=True,min_cluster_size=24,core_dist_n_jobs=-1,gen_min_span_tree=True,cluster_selection_method='leaf').fit(data_gaia_selected)

In [ ]:
np.unique(clusterer_selected.labels_)

In [ ]:
clusterer_selected.minimum_spanning_tree_.plot()